# 31 - The Agent Protocol Stack

## Scenario: Northstar Incident Response

In the previous modules, we built individual components of an enterprise agent system. Now, we explore how autonomous agents communicate securely across system boundaries using standard **Protocols**.

The AI landscape relies heavily on several emerging protocols:
1. **Model Context Protocol (MCP)**: Connecting agents to Tools & Context.
2. **Agent Protocol (AIEF)**: Executing and interacting with an Agent lifecycle.
3. **Agent-to-Agent (A2A)**: Delegating tasks between autonomous agents.
4. **Realtime & GenUI**: Streaming multimodal interfaces directly to users.

This notebook demonstrates the underlying structures and schemas that make this possible.



In [1]:
# 1. Setup
# Let's import the necessary libraries. 
# We'll use standard Python dataclasses to model the schemas of the various protocols.
import json
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field

print("✅ Protocol libraries initialized.")



✅ Protocol libraries initialized.



## 1. Model Context Protocol (MCP)

**Standard by Anthropic (2024)**

MCP standardizes how an AI application (the Host) connects to external Data Sources and Tools (the Servers) using JSON-RPC. Let's see what a server looks like conceptually.


In [1]:
# Note: In a real environment, you would run this in a separate server process.
# We will define a mock FastMCP server for querying the Northstar Jira database.

@dataclass
class MockMCPTool:
    name: str
    description: str
    input_schema: Dict[str, Any]

class MockFastMCP:
    def __init__(self, name: str):
        self.name = name
        self.tools = []
    
    def tool(self):
        def decorator(func):
            schema = {
                "type": "object",
                "properties": {"ticket_id": {"type": "string"}},
                "required": ["ticket_id"]
            }
            self.tools.append(MockMCPTool(name=func.__name__, description=func.__doc__, input_schema=schema))
            return func
        return decorator

# --- Building the MCP Server ---
jira_mcp = MockFastMCP("Northstar Jira Server")

@jira_mcp.tool()
def query_ticket(ticket_id: str) -> str:
    """Query the enterprise Jira database for an incident ticket."""
    return f"Ticket {ticket_id}: Payment gateway timeout error."

print(f"🔧 Starting MCP Server: {jira_mcp.name}")
print("Exposed Tools:")
for t in jira_mcp.tools:
    print(f" - {t.name}: {t.description}")



🔧 Starting MCP Server: Northstar Jira Server
Exposed Tools:
 - query_ticket: Query the enterprise Jira database for an incident ticket.



### The MCP Client

The Agent needs to connect to the MCP server. Usually, this is done via `stdio` (local process) or `SSE` (network HTTP). Let's simulate an agent calling the `query_ticket` tool over JSON-RPC.


In [1]:
# --- Simulating the MCP Client Request ---
mcp_request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "query_ticket",
        "arguments": {"ticket_id": "INC-8891"}
    }
}

print(f"Agent sends JSON-RPC request to MCP Server:\n{json.dumps(mcp_request, indent=2)}\n")

# Server processes the request...
result = query_ticket(**mcp_request["params"]["arguments"])

mcp_response = {
    "jsonrpc": "2.0",
    "id": 1,
    "result": {
        "content": [{"type": "text", "text": result}],
        "isError": False
    }
}

print(f"MCP Server responds:\n{json.dumps(mcp_response, indent=2)}")



Agent sends JSON-RPC request to MCP Server:
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "tools/call",
  "params": {
    "name": "query_ticket",
    "arguments": {
      "ticket_id": "INC-8891"
    }
  }
}

MCP Server responds:
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "Ticket INC-8891: Payment gateway timeout error."
      }
    ],
    "isError": false
  }
}



## 2. Agent Protocol (AI Engineer Foundation)

While MCP is for *tools*, the **Agent Protocol** standardizes how a client *orchestrates an agent*. It defines REST endpoints for **Runs**, **Tasks**, and **Steps**. This is widely used in sandboxed environments (like E2B).


In [1]:
# Modeling the Agent Protocol REST schemas
@dataclass
class TaskRequestBody:
    input: str
    additional_input: Optional[Dict[str, Any]] = None

@dataclass
class Task:
    task_id: str
    input: str
    artifacts: List[Dict] = field(default_factory=list)

@dataclass
class Step:
    task_id: str
    step_id: str
    name: str
    status: str # "created", "running", "completed"
    output: Optional[str] = None

# Mocking the REST API interaction
task_req = TaskRequestBody(input="Analyze deployment logs for incident INC-8891.")
task_record = Task(task_id="tsk_99x", input=task_req.input)

print("🌐 [POST /ap/v1/agent/tasks]")
print(f"Client creates task: {task_record.task_id}\n")

step = Step(task_id=task_record.task_id, step_id="stp_1", name="Analyze", status="running")
print(f"🌐 [POST /ap/v1/agent/tasks/{task_record.task_id}/steps]")
print(f"Client executes step: {step.step_id} - Status: {step.status}")

step.status = "completed"
step.output = "Log analysis complete. Found OOM Error in payment pod."
print(f"Step completed. Output: {step.output}")



🌐 [POST /ap/v1/agent/tasks]
Client creates task: tsk_99x

🌐 [POST /ap/v1/agent/tasks/tsk_99x/steps]
Client executes step: stp_1 - Status: running
Step completed. Output: Log analysis complete. Found OOM Error in payment pod.



## 3. Agent-to-Agent (A2A) Delegation

What happens when an agent needs to delegate work to another autonomous agent? The **A2A Protocol** relies on **Agent Cards** (a manifest advertising capabilities) and standardized Task Delegation.


In [1]:
@dataclass
class AgentCard:
    agent_id: str
    description: str
    authentication_required: bool
    supported_task_schema: Dict[str, Any]

# The Remote Specialist publishes its card
release_specialist = AgentCard(
    agent_id="northstar-release-agent",
    description="Analyzes Kubernetes pod deployments and logs.",
    authentication_required=True,
    supported_task_schema={"type": "object", "properties": {"namespace": {"type": "string"}}}
)

print(f"🪪 Discovered Agent Card: {release_specialist.agent_id}")

# The Coordinator Agent creates a Delegation Payload
delegation_payload = {
    "protocol": "a2a/v1",
    "source_agent": "incident-coordinator-1",
    "target_agent": release_specialist.agent_id,
    "task": {
        "objective": "Check deployment health",
        "inputs": {"namespace": "payments-prod"},
        "budget_limit_usd": 0.50
    }
}

print(f"\n🤝 A2A Task Delegation Payload:\n{json.dumps(delegation_payload, indent=2)}")
print("\nNote: A2A enables discovery and coordination, but the target agent must independently verify authorization before accepting the task!")



🪪 Discovered Agent Card: northstar-release-agent

🤝 A2A Task Delegation Payload:
{
  "protocol": "a2a/v1",
  "source_agent": "incident-coordinator-1",
  "target_agent": "northstar-release-agent",
  "task": {
    "objective": "Check deployment health",
    "inputs": {
      "namespace": "payments-prod"
    },
    "budget_limit_usd": 0.5
  }
}

Note: A2A enables discovery and coordination, but the target agent must independently verify authorization before accepting the task!



## 4. UI & Presentation Layer

Finally, agents interact with humans. 
- **AG-UI & GenUI**: Stream native React components (JSON schemas) instead of raw text.
- **WebRTC / Live APIs**: Handle bi-directional, low-latency Audio/Video streaming.


In [1]:
# A conceptual Generative UI JSON payload
genui_payload = {
    "type": "ui_component",
    "component_id": "ApprovalCard",
    "props": {
        "title": "Incident Mitigation Proposal",
        "details": "Restart payment pod in us-east-1.",
        "actions": ["Approve", "Reject"]
    }
}

print("💻 Streaming UI Component to Frontend Renderer:")
print(json.dumps(genui_payload, indent=2))
print("\nThe frontend receives this and renders a native React Approval Card safely, without executing arbitrary code.")



💻 Streaming UI Component to Frontend Renderer:
{
  "type": "ui_component",
  "component_id": "ApprovalCard",
  "props": {
    "title": "Incident Mitigation Proposal",
    "details": "Restart payment pod in us-east-1.",
    "actions": [
      "Approve",
      "Reject"
    ]
  }
}

The frontend receives this and renders a native React Approval Card safely, without executing arbitrary code.



## Checkpoint

**1. Which protocol provides a standard REST API to track Runs, Tasks, and Steps of an agent's execution?**
- A) Model Context Protocol (MCP)
- B) Agent Protocol
- C) WebRTC
- D) Generative UI

<details>
<summary>Answer</summary>
<b>B</b>. The Agent Protocol standardizes the client-to-agent lifecycle and execution.
</details>

